# 尖峰負載（老師版）

老師對照用，不發給學生。格子跟 `尖峰負載.ipynb` 一樣：1 資料、2 前處理、3 訓練、4 你的判斷。

這是線性回歸的備案委託，不隨實驗安裝。工作本、`委託.html`、資料在 repo 的 `linear-regression/備案/尖峰負載/`，要用時把整個資料夾發給學生。

在 repo 的 `linear-regression/teacher/` 直接打開就能跑，放進學生的 `尖峰負載/` 資料夾也能跑。只用 numpy、matplotlib，寫法跟 `線性回歸.ipynb` 一樣：`loadData`、z-score、Cost、梯度下降都自己寫。

模型：

$$
y = w_1 \max(0,\ \text{最高溫} - 20) + w_2\, x_{\text{放假}} + b
$$

$\max(0,\ \text{最高溫} - 20)$ 叫「熱度」：20°C 以下是 0，熱了才往上加，做出曲棍球棒的形狀。$x_{\text{放假}}$ 是 0/1，放假那天是 1。

| | 參考值（全部 577 天） | 這本（隨機 80% 訓練） |
|---|---|---|
| $w_1$（熱度） | 58.52 | 約 58.57 |
| $w_2$（放假） | −433.13 | 約 −429.58 |
| $b$ | 2990.43 | 約 2990.66 |
| 只用最高溫（直線） | $w=50.99$，$b=1911.97$ | 約 $52.71$、$1870.5$ |

隨機切 20% 當測試時，熱度的 $w$ 大約在 58 到 60，放假的 $w$ 大約在 −450 到 −427。

學生如果不用 20°C，而用 18 到 26 之間的其他轉折點，結果都差不多（參考值的 Cost 在 22°C 最低）。學生用最高溫平方也可以，效果和熱度差不多。


## 1 資料


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

def loadData(filename, dtype=float):
    try:
        here = Path(__vsc_ipynb_file__).resolve().parent
    except NameError:
        here = Path.cwd()
    # 跟工作本放在一起：資料在旁邊的 data/
    # 留在 repo 的 linear-regression/teacher/：資料在 linear-regression/備案/尖峰負載/data/
    folders = [here / "data", here.parent / "備案" / "尖峰負載" / "data"]
    folder = next((f for f in folders if (f / filename).is_file()), folders[0])
    return np.loadtxt(
        folder / filename,
        delimiter=",",
        skiprows=1,
        encoding="utf-8",
        dtype=dtype,
    )

# 日期、星期、假日名稱是文字，先全部當文字讀進來，要用的欄再轉成數字
rows = loadData("taipower_peak.csv", dtype=str)
columns = ["日期", "星期", "尖峰負載", "最高溫", "最低溫", "雨量", "放假", "假日名稱"]
print("筆數", len(rows))
for j, name in enumerate(columns):
    print(f"{name} 空白 {(rows[:, j] == '').sum()} 格")

load = rows[:, 2].astype(float)     # 尖峰負載（萬瓩），要預測的
tmax = rows[:, 3].astype(float)     # 台北最高溫（℃）
tmin = rows[:, 4].astype(float)     # 台北最低溫（℃）
rain = rows[:, 5].astype(float)     # 台北雨量（毫米）
day_off = rows[:, 6].astype(float)  # 放假（1為放假）
print("尖峰負載 最小", load.min(), "最大", load.max())
print("放假天數", int(day_off.sum()), "上班天數", int((day_off == 0).sum()))

# 上班日和放假日分兩種顏色：兩條帶子上下錯開；20°C 以下幾乎平的，熱了才往上衝
plt.scatter(tmax[day_off == 0], load[day_off == 0], s=8, alpha=0.5, label="workday")
plt.scatter(tmax[day_off == 1], load[day_off == 1], s=8, alpha=0.5, label="day off")
plt.xlabel("Taipei max temp (C)")
plt.ylabel("Peak load (10 MW)")
plt.title("Peak load vs. max temp")
plt.legend()
plt.show()


## 2 前處理

不用的欄：

- **最低溫、雨量**：先不放，在「你的判斷」再檢查放進去有沒有差。
- **星期**：放假已經包含週末，不另外放。
- **假日名稱**：是文字，放假已經包含國定假日。

資料沒有空白，577 天全部留下。

委託資料裡沒有民生用電、工業用電：它們是同一天的用電量，幾乎就是答案，放了就是資料洩漏。


In [ ]:
# 特徵 1：熱度 = max(0, 最高溫 - 20)。20°C 以下是 0，熱了才往上加，做出曲棍球棒的形狀
# 特徵 2：放假，本來就是 0/1
heat = np.maximum(0, tmax - 20)
X_all = np.column_stack([heat, day_off])
y_all = load
print("前處理後", X_all.shape)

# 打亂後 80% 訓練、20% 測試
# seed 3 是典型的一種切法：測試 Cost 跟訓練 Cost 差不多，係數也接近參考值
rng = np.random.default_rng(3)
order = rng.permutation(len(X_all))
cut = int(len(X_all) * 0.8)
train, test = order[:cut], order[cut:]
X_train, y_train = X_all[train], y_all[train]
X_test, y_test = X_all[test], y_all[test]
print("訓練", X_train.shape, "測試", X_test.shape)

# 熱度大約 0 到 18，放假是 0/1，尺度不同，用 z-score 縮放
# 平均和標準差只用訓練資料算，測試資料套用同一組
def zscore_normalize_features(X):
    mu = np.mean(X, axis=0)
    sigma = np.std(X, axis=0)
    return (X - mu) / sigma, mu, sigma

X_train_norm, mu, sigma = zscore_normalize_features(X_train)
X_test_norm = (X_test - mu) / sigma
print("訓練資料的 mu", mu.round(3), "sigma", sigma.round(3))


## 3 訓練


In [ ]:
def compute_cost_multi(X, y, w, b):
    m = X.shape[0]
    prediction = np.dot(X, w) + b
    return np.sum((prediction - y) ** 2) / (2 * m)

def compute_gradient_multi(X, y, w, b):
    m = X.shape[0]
    errors = np.dot(X, w) + b - y
    dj_dw = np.dot(X.T, errors) / m
    dj_db = np.sum(errors) / m
    return dj_dw, dj_db

def gradient_descent_multi(X, y, alpha, num_iters):
    w = np.zeros(X.shape[1])
    b = 0.0
    history = []
    for _ in range(num_iters):
        dj_dw, dj_db = compute_gradient_multi(X, y, w, b)
        w = w - alpha * dj_dw
        b = b - alpha * dj_db
        history.append(compute_cost_multi(X, y, w, b))
    return w, b, history

def to_original_scale(w, b, mu, sigma):
    # 縮放後學到的 w、b 換回原始單位，才能跟參考值比，也比較好解釋
    return w / sigma, b - np.sum(w * mu / sigma)

w_norm, b_norm, history = gradient_descent_multi(X_train_norm, y_train, alpha=0.1, num_iters=1000)
w, b = to_original_scale(w_norm, b_norm, mu, sigma)
print("換回原始單位的 w =", w.round(2), "b =", round(float(b), 2))
print("參考值約 w=[58.52, -433.13], b=2990.43（用全部資料算的，這裡只用 80%，會差一點）")

cost_train = compute_cost_multi(X_train_norm, y_train, w_norm, b_norm)
cost_test = compute_cost_multi(X_test_norm, y_test, w_norm, b_norm)
# Cost = 平均平方誤差 / 2，所以 sqrt(2 * Cost) 是「平均大約差多少萬瓩」
print(f"訓練 Cost {cost_train:.0f}（約差 {np.sqrt(2 * cost_train):.0f} 萬瓩）")
print(f"測試 Cost {cost_test:.0f}（約差 {np.sqrt(2 * cost_test):.0f} 萬瓩）")

plt.plot(history)
plt.xlabel("epoch")
plt.ylabel("Cost")
plt.title("Cost should fall, then flatten")
plt.show()


In [ ]:
# 對照：同一組切法，換不同的特徵，看測試 Cost
def fit_and_test(X):
    Xn, m_, s_ = zscore_normalize_features(X[train])
    w_n, b_n, _ = gradient_descent_multi(Xn, y_all[train], alpha=0.1, num_iters=1000)
    w_o, b_o = to_original_scale(w_n, b_n, m_, s_)
    return w_o, b_o, compute_cost_multi((X[test] - m_) / s_, y_all[test], w_n, b_n)

for label, X in [
    ("只用最高溫（直線）", tmax[:, None]),
    ("最高溫 + 放假", np.column_stack([tmax, day_off])),
    ("熱度 + 放假（這本）", X_all),
]:
    w_o, b_o, c = fit_and_test(X)
    print(f"{label:12s} 測試 Cost {c:7.0f}  w {w_o.round(2)}  b {b_o:.1f}")


In [ ]:
# 貼合程度：測試資料的點，加上上班日、放假日各一條曲棍球棒
t_line = np.linspace(tmax.min(), tmax.max(), 200)
for off, color, label in [(0, "tab:blue", "workday"), (1, "tab:orange", "day off")]:
    mask = X_test[:, 1] == off
    plt.scatter(tmax[test][mask], y_test[mask], s=12, c=color, alpha=0.6, label=f"{label} test days")
    plt.plot(t_line, w[0] * np.maximum(0, t_line - 20) + w[1] * off + b, c=color)
plt.xlabel("Taipei max temp (C)")
plt.ylabel("Peak load (10 MW)")
plt.title("Test days and fitted curves")
plt.legend()
plt.show()

# 測試資料的實際負載對預測負載，點貼著虛線表示估得準
pred_test = np.dot(X_test_norm, w_norm) + b_norm
lims = [min(y_test.min(), pred_test.min()), max(y_test.max(), pred_test.max())]
plt.scatter(y_test, pred_test, s=12, alpha=0.6)
plt.plot(lims, lims, "k--", label="perfect")
plt.xlabel("Actual peak load (10 MW)")
plt.ylabel("Predicted peak load (10 MW)")
plt.title("Test days: actual vs. predicted")
plt.legend()
plt.show()


<div style="background:#fff6e0;border-left:6px solid #e09b00;padding:12px 16px;border-radius:4px;color:#1a1a1a;">

<h3 style="margin-top:0;color:#1a1a1a;">4 你的判斷（參考答案，學生要自己寫）</h3>
<ol style="color:#1a1a1a;list-style:decimal;padding-left:1.4em;">
<li style="color:#1a1a1a;"><b>天氣變熱時，用電怎麼走？冷天和熱天一樣嗎？</b><br>
20°C 以下，用電幾乎不跟著氣溫變。20°C 以上，最高溫每多 1°C，尖峰負載大約多 59 萬瓩，主要是冷氣。只用一條直線（R² 約 0.60）會低估夏天、高估冬天；改成熱度以後，測試 Cost 從約 42800 降到約 12600。</li>
<li style="color:#1a1a1a;"><b>放假那天，用電差多少？</b><br>
同樣的氣溫，放假那天大約少 430 萬瓩。工廠和辦公室在放假時用電少，所以差這麼多。</li>
<li style="color:#1a1a1a;"><b>最低溫、雨量有沒有幫助？</b>（下一格各加一次看看，題目沒直接問，批改時要知道）<br>
<b>雨量</b>：測試 Cost 和係數幾乎不變，沒有影響。<br>
<b>最低溫</b>：測試 Cost 從約 12600 降到約 9700，但熱度的 w 從約 59 變成約 32。最高溫和最低溫的相關係數約 0.89，兩欄講的是重疊的事，「熱的影響」被最低溫分走一部分。方向不變：越熱，用電越多。熱的晚上冷氣不關，所以最低溫也有用。</li>
<li style="color:#1a1a1a;"><b>估得最不準的是哪幾天？</b><br>
實際比預測低很多的日子，多數在春節前後（2026-02-13、2026-02-15、2025-02-01）：放假欄只有 0/1，看不出春節比一般週末更少人上班。2025-05-01 勞動節，行政機關上班、很多公司放假，所以放假欄是 0，但用電像半個假日。2025-07-07、2025-11-12 可以讓學生自己查當天新聞，例如有沒有颱風。</li>
</ol>

</div>


In [ ]:
# 同一組切法，各加一次最低溫、雨量
for label, X in [
    ("熱度+放假", X_all),
    ("再加最低溫", np.column_stack([X_all, tmin])),
    ("再加雨量", np.column_stack([X_all, rain])),
]:
    w_o, b_o, c = fit_and_test(X)
    print(f"{label:8s} 測試 Cost {c:7.0f}  熱度的 w {w_o[0]:6.2f}  放假的 w {w_o[1]:7.2f}")
print("最高溫和最低溫的相關係數", round(float(np.corrcoef(tmax, tmin)[0, 1]), 2))

# 估得最差的幾天：實際比預測低很多
pred_all = np.dot((X_all - mu) / sigma, w_norm) + b_norm
worst = np.argsort(load - pred_all)[:6]
for i in worst:
    print(rows[i, 0], "星期" + rows[i, 1], rows[i, 7], f"實際 {load[i]:.0f}，預測 {pred_all[i]:.0f}")
